In [3]:
import os, glob, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping as XGBEarlyStopping


In [4]:
BASE_PATH = os.path.join(os.environ.get('USERPROFILE', ''), 'Downloads', 'fraud_stream_parquet')
OUT_DIR   = os.path.join(os.environ.get('USERPROFILE', ''), 'Downloads', 'baseline1_results')
os.makedirs(OUT_DIR, exist_ok=True)

FEATURES = [
    'TX_AMOUNT', 'TX_TIME_DAYS', 'TX_TIME_SECONDS',
    'x_customer_id','y_customer_id','mean_amount','std_amount','mean_nb_tx_per_day',
    'x_terminal_id','y_terminal_id'
]
TARGET = 'TX_FRAUD'

PRE_MONTHS = 4                 # meses para pretraining
VAL_DAYS_LAST_MONTH = 14       # días del mes 4 para calibrar umbral F1
START_DATE  = "2025-01-01" 
GRANULARITY = 'week'          # 'month' (recomendado). Si quieres diario: 'day'.
TIMELINE_FILE = os.path.join(BASE_PATH, 'timeline.parquet')  # opcional; si no existe, TTA90 usa un solo régimen

# TTA90: ventanas para "plateau" por régimen (si GRANULARITY='month', usa 2; si 'day', usa 10)
PLATEAU_WINDOW = 4 #2 if GRANULARITY == 'month' else 10
TTA_TARGET_FRAC = 0.90  # 90%

def compute_base_week(eval_df: pd.DataFrame) -> int:
    base_week = int(eval_df['TX_TIME_DAYS'].min() // 7) + 1
    base_day = int(eval_df['TX_TIME_DAYS'].min())  # Día absoluto desde START_DATE
    base_date = pd.to_datetime(START_DATE) + pd.Timedelta(days=base_day)
    print(f"Semana base: {base_week}, Fecha: {base_date.strftime('%Y-%m-%d')}")
    # primera semana presente en el df de evaluación
    return base_week


In [5]:
def load_all_parquet_by_year(base_path: str) -> pd.DataFrame:
    year_dirs = sorted(glob.glob(os.path.join(base_path, "TX_YEAR=*")))
    dfs = []
    for ydir in year_dirs:
        files = sorted(glob.glob(os.path.join(ydir, "*.parquet")))
        if not files:
            continue
        dfs.append(pd.concat([pd.read_parquet(f) for f in files], ignore_index=True))
    if not dfs:
        raise RuntimeError(f"No se encontraron Parquet en {base_path}")
    df = pd.concat(dfs, ignore_index=True)
    # Orden temporal sin TX_DATETIME
    df = df.sort_values(['TX_YEAR','TX_MONTH','TX_DAY','TX_TIME_SECONDS'], kind='mergesort').reset_index(drop=True)
    #print(df.head())
   # print(df.columns.tolist())
    return df


def add_time_indexes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    first_year = int(df['TX_YEAR'].min())
    df['_month_idx'] = (df['TX_YEAR'] - first_year) * 12 + df['TX_MONTH']  # 1..24

    df['_week_idx']  = (df['TX_TIME_DAYS'] // 7).astype(int) + 1  
    # índice de "día absoluto" desde el inicio (por si usas diario)
    # aproximación: mapeo (year,month,day) a orden relativo creciente
    df['_day_abs'] = (df['_month_idx'] - 1) * 31 + df['TX_DAY']  # suficiente para ordenar de forma estable
    return df

def group_chunks(df: pd.DataFrame, granularity='month', start_date_str=START_DATE):
    if granularity == 'month':
        for (y, m), g in df.groupby(['TX_YEAR','TX_MONTH'], sort=True):
            yield (int(y), int(m)), g, f"{int(y)}-{int(m):02d}"

    elif granularity == 'day':
        for (y, m, d), g in df.groupby(['TX_YEAR','TX_MONTH','TX_DAY'], sort=True):
            yield (int(y), int(m), int(d)), g, f"{int(y)}-{int(m):02d}-{int(d):02d}"

    elif granularity == 'week':
        d0 = pd.to_datetime(start_date_str)
        for wk, g in df.groupby('_week_idx', sort=True):
            wk = int(wk)
            wstart = d0 + pd.Timedelta(days=(wk-1)*7)
            iso = wstart.isocalendar()  # (year, week, weekday)
            label = f"{int(iso.year)}-W{int(iso.week):02d}"
            yield wk, g, label
    else:
        raise ValueError("granularity debe ser 'month' | 'week' | 'day'.")


In [6]:
def split_pretrain_val(df: pd.DataFrame, pre_months=4, val_days_last_month=14):
    df = df.copy()
    m = pre_months
    pre_mask = (df['_month_idx'] < m)              # meses 1..3
    last_month_mask = (df['_month_idx'] == m)      # mes 4
    # últimas 2 semanas del mes 4 como val
    dmax = int(df.loc[last_month_mask, 'TX_DAY'].max())
    cutoff = max(1, dmax - val_days_last_month + 1)
    train_last = last_month_mask & (df['TX_DAY'] < cutoff)
    val_last   = last_month_mask & (df['TX_DAY'] >= cutoff)
    train_mask = pre_mask | train_last
    val_mask   = val_last
    return train_mask, val_mask


In [7]:
class NoOpScaler:
    def fit(self, X): return self
    def fit_transform(self, X): return X
    def transform(self, X): return X

In [8]:
def fit_baseline(train_df, val_df, features, target='TX_FRAUD'):
    """
    Baseline con XGBoost para datos desbalanceados.
    - Calcula scale_pos_weight = #neg / #pos
    - No usamos escalado (árboles), devolvemos un NoOpScaler para mantener el flujo.
    - Calibra umbral para F1 en el val del mes 4.
    """
    # ====== Datos ======
    Xtr = train_df[features].astype('float32').values
    ytr = train_df[target].astype('int32').values
    Xv  = val_df[features].astype('float32').values
    yv  = val_df[target].astype('int32').values

    # ====== Peso de clase (desbalance) ======
    pos = int(ytr.sum())
    neg = int(len(ytr) - pos)
    scale_pos_weight = float(neg / max(1, pos))

    # ====== Modelo XGBoost ======
    clf = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',   # podrías usar 'aucpr' si pasas eval_set
        tree_method='hist',      # cambia a 'gpu_hist' si tienes GPU
        verbosity=0
    )

    # ====== Entrenamiento ======
    t0 = time.perf_counter()
    clf.fit(Xtr, ytr)
    train_time_s = time.perf_counter() - t0

    # ====== Validación para umbral F1 ======
    t0 = time.perf_counter()
    pv = clf.predict_proba(Xv)[:, 1] #Probabilidad del set de validacion en el mes 4
    inf_time_val_s = time.perf_counter() - t0

    ap_ref = float(average_precision_score(yv, pv))
    prec, rec, thr = precision_recall_curve(yv, pv) #Cálculo de precisión y recall para varios umbrales
    f1s = (2 * prec * rec) / (prec + rec + 1e-12) #Cálculo de F1 para varios umbrales
    best_idx = int(np.nanargmax(f1s[:-1])) if len(thr) > 0 else 0
    best_thr = float(thr[best_idx]) if len(thr) > 0 else 0.5

    print(f"Mejor umbral F1 en validación: {best_thr:.4f} (F1={f1s[best_idx]:.4f}, Prec={prec[best_idx]:.4f}, Rec={rec[best_idx]:.4f})")
    f1_ref = float(f1_score(yv, (pv >= best_thr).astype(int)))
    print(f"AP validación: {ap_ref:.4f}, F1 validación: {f1_ref:.4f}")
    # No usamos escalado con árboles, devolvemos un "scaler" identidad
    scaler = NoOpScaler()

    return scaler, clf, best_thr, ap_ref, f1_ref, train_time_s, (inf_time_val_s / max(1, len(yv)))


In [9]:
def load_timeline_month_states(timeline_path, df_all, granularity='month'):
    """
    Devuelve lista de tuplas (start_idx, end_idx, state_str) por régimen.
    Si no hay timeline, asume un único régimen (todo el periodo).
    """
    # Mapeo etiqueta->índice en la serie de métricas
    if granularity != 'month':
        # Para granularidad diaria, necesitarías timeline diario (S1,S2,S3 por día)
        # Este script centra TTA90 mensual porque tus escenarios están definidos en meses/bimestres.
        return [(0, None, "ALL")]  # todo el periodo

    # Si no existe timeline -> todo el periodo
    if not os.path.exists(timeline_path):
        return [(0, None, "ALL")]

    tl = pd.read_parquet(timeline_path)
    # timeline esperado con columnas: TX_TIME_DAYS, S1,S2,S3 (0/1), state (string)
    # agregamos por mes (modo más frecuente del estado en el mes)
    # necesitamos mapear mes: usaremos df_all para (year, month) por day_idx
    aux = df_all[['TX_YEAR','TX_MONTH','TX_DAY']].drop_duplicates().copy()
    # aproximación: convertir (year,month,day) a day_idx relativo al start_date original si lo tuviéramos;
    # como no lo tenemos aquí, asumimos que timeline.TX_TIME_DAYS arranca en 0 y que df_all está ordenado.
    # Para emparejar, sacamos el mes desde TX_TIME_DAYS basándonos en el primer año/mes/día del df.
    # Mas simple: construimos un DataFrame de meses con rango de TX_TIME_DAYS (si no es trivial, caemos a "ALL").
    if 'TX_TIME_DAYS' not in tl.columns:
        return [(0, None, "ALL")]

    # mes por TX_TIME_DAYS usando df_all (aprox): tomamos el valor mínimo de TX_TIME_DAYS para cada (Y,M)
    # y asignamos días dentro de ese bloque.
    # Si no hay correspondencia exacta, reducimos a "ALL".
    try:
        # Mapeo día->(Y,M)
        # Creamos un mapping rápido usando el df_all (ordenado) tomando pares (day_abs -> month label)
        df_sorted = df_all.sort_values(['TX_YEAR','TX_MONTH','TX_DAY']).copy()
        df_sorted['_day_seq'] = np.arange(len(df_sorted))
        # day_seq no es TX_TIME_DAYS, pero guarda orden; usamos cortes por mes para generar ranges
        month_groups = df_sorted.groupby(['TX_YEAR','TX_MONTH'])['_day_seq'].agg(['min','max']).reset_index()
        # timeline por TX_TIME_DAYS -> lo llevamos a day_seq asumiendo igualdad de longitud total
        max_tl = int(tl['TX_TIME_DAYS'].max())
        max_seq = int(df_sorted['_day_seq'].max())
        # Escala lineal day_idx ~ TX_TIME_DAYS (aprox)
        tl['_day_seq'] = (tl['TX_TIME_DAYS'] * (max_seq / max(1, max_tl))).round().astype(int).clip(0, max_seq)
        # asigna mes por _day_seq usando interval merge
        month_ranges = []
        for _, r in month_groups.iterrows():
            month_ranges.append( (int(r['min']), int(r['max']), int(r['TX_YEAR']), int(r['TX_MONTH'])) )
        month_ranges = sorted(month_ranges)

        def seq_to_ym(s):
            # busca en month_ranges
            # (búsqueda lineal está bien: solo 24 meses)
            for mn, mx, y, m in month_ranges:
                if mn <= s <= mx:
                    return y, m
            return None, None

        tl[['TL_YEAR','TL_MONTH']] = tl['_day_seq'].apply(lambda s: pd.Series(seq_to_ym(s)))

        tlm = tl.dropna(subset=['TL_YEAR','TL_MONTH']).copy()
        tlm['state'] = tlm.apply(lambda r: f"S1={int(r.get('S1',0))}|S2={int(r.get('S2',0))}|S3={int(r.get('S3',0))}", axis=1)

        # estado por mes = moda
        month_state = (tlm
            .groupby(['TL_YEAR','TL_MONTH'])['state']
            .agg(lambda s: s.value_counts().idxmax())
            .reset_index()
            .sort_values(['TL_YEAR','TL_MONTH'])
        )

        # convertir a segmentos contiguos por estado
        labels = [f"{int(y)}-{int(m):02d}" for y,m in zip(month_state['TL_YEAR'], month_state['TL_MONTH'])]
        states = month_state['state'].tolist()

        segments = []
        if labels:
            start = 0
            for i in range(1, len(labels)):
                if states[i] != states[i-1]:
                    segments.append( (start, i-1, states[i-1]) )
                    start = i
            segments.append( (start, len(labels)-1, states[-1]) )
        else:
            segments = [(0, None, "ALL")]
        # devolvemos índices relativos al primer mes presente en métricas
        return segments
    except Exception:
        # si algo no calza, usamos régimen único
        return [(0, None, "ALL")]


In [10]:
def compute_max_drawdown_percent(series_ap):
    running_max = -np.inf
    max_dd = 0.0
    for ap in series_ap:
        running_max = max(running_max, ap)
        if running_max > 0:
            dd = (ap - running_max) / running_max
            max_dd = min(max_dd, dd)
    return 100.0 * max_dd  # negativo o 0


In [11]:
def drawdown_series(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula running max y drawdown% semana a semana sobre AUPRC.
    Devuelve df con columnas: pos, label, AUPRC, run_max, drawdown_pct.
    """
    out = metrics_df[['pos','label','AUPRC']].copy()
    a = out['AUPRC'].values
    run_max = np.maximum.accumulate(np.nan_to_num(a, nan=0.0))
    # dd% = (a - run_max)/run_max * 100 (negativo o 0). Si run_max==0 -> 0.
    dd = []
    for i in range(len(a)):
        rm = run_max[i]
        if rm <= 0:
            dd.append(0.0)
        else:
            dd.append( (a[i] - rm) / rm * 100.0 )
    out['run_max'] = run_max
    out['drawdown_pct'] = dd
    return out

def plot_drawdown_weekly(dd_df: pd.DataFrame, out_dir: str, month_markers=None, bimonth_markers=None):
    x = dd_df['pos'].values
    plt.figure()
    plt.plot(x, dd_df['drawdown_pct'])
    if month_markers:
        for wk, _ in month_markers: plt.axvline(wk, color='gray', alpha=0.15)
        xs, labs = zip(*month_markers); plt.xticks(xs, labs, rotation=45, ha='right')
    if bimonth_markers:
        for wk, _ in bimonth_markers: plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)
    plt.xlabel('Semanas'); plt.ylabel('Drawdown % (≤ 0)')
    plt.title('Olvido/estabilidad: Drawdown de AUPRC (semanal)')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'drawdown_semana.png')); plt.close()

def compute_max_drawdown_percent_from_series(dd_df: pd.DataFrame) -> float:
    # el mínimo (más negativo) de la serie
    return float(np.nanmin(dd_df['drawdown_pct'].values))


In [12]:
def compute_bimonth_markers(eval_df: pd.DataFrame, pre_months: int, base_week: int):
    # índices de mes relativos al primer año del eval_df
    fy = int(eval_df['TX_YEAR'].min())
    eval_df = eval_df.copy()
    eval_df['_midx'] = (eval_df['TX_YEAR'] - fy) * 12 + eval_df['TX_MONTH']  # 1..N dentro del eval_df

    start_m = int(eval_df['_midx'].min())  # debería ser 5 si el pretraining es 1..4
    end_m   = int(eval_df['_midx'].max())

    # Fronteras entre bloques bimestrales: M0+2, M0+4, … (no dibujamos en M0)
    M0 = start_m
    boundaries = list(range(M0 + 2, end_m + 1, 2))

    marks = []
    for b in boundaries:
        y = fy + (b - 1) // 12
        m = (b - 1) % 12 + 1
        # 1er día (en eval_df) de ese mes b
        rows = eval_df[(eval_df['TX_YEAR']==y) & (eval_df['TX_MONTH']==m)]
        if rows.empty:
            continue
        day_min = int(rows['TX_TIME_DAYS'].min())
        wk_abs  = day_min // 7 + 1
        wk_rel  = int(wk_abs - base_week + 1)
        marks.append((wk_rel, f"M{m:02d}"))
    return marks


In [13]:
def plot_series(metrics_df, out_dir, month_markers=None, bimonth_markers=None):
    # AUPRC en el tiempo
    x = metrics_df['pos']

    def _apply_month_ticks():
        if month_markers:
            for wk, _ in month_markers:
                plt.axvline(wk, color='gray', alpha=0.15)  # líneas suaves por mes
            xs, labs = zip(*month_markers)
            plt.xticks(xs, labs, rotation=45, ha='right')

    def _apply_bimonth_lines():
        if bimonth_markers:
            for wk, _ in bimonth_markers:
                plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)  # punteada


    plt.figure()
    plt.plot(x, metrics_df['AUPRC'])
    _apply_month_ticks()
    _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('AUPRC'); plt.title('AUPRC por semana (baseline #1)')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'auprc_semana.png')); plt.close()
 
    # F1 semanal
    plt.figure()
    plt.plot(x, metrics_df['F1'])
    _apply_month_ticks()
    _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('F1 (umbral fijo)'); plt.title('F1 por semana (baseline #1)')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'f1_semana.png')); plt.close()

    # Drawdown AUPRC
    running_max = np.maximum.accumulate(metrics_df['AUPRC'].values)
    plt.figure()
    plt.plot(x, metrics_df['AUPRC'], label='AUPRC (semanal)')
    plt.plot(x, running_max, label='Pico acumulado', linestyle='--')
    _apply_month_ticks()
    _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('AUPRC'); plt.title('AUPRC vs. Pico acumulado (semanal)')
    plt.legend()
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'auprc_drawdown_semana.png')); plt.close()

    # Latencia semanal
    plt.figure()
    plt.plot(x, metrics_df['infer_ms_per_tx'])
    _apply_month_ticks()
    _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('ms por transacción'); plt.title('Latencia de inferencia (semanal)')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, 'latencia_semana.png')); plt.close()



In [14]:
def compute_month_markers(eval_df: pd.DataFrame, base_week: int):
    marks = []
    months = (eval_df[['TX_YEAR','TX_MONTH']]
              .drop_duplicates()
              .sort_values(['TX_YEAR','TX_MONTH']))
    for _, r in months.iterrows():
        y, m = int(r['TX_YEAR']), int(r['TX_MONTH'])
        day_min = int(eval_df[(eval_df['TX_YEAR']==y) & (eval_df['TX_MONTH']==m)]['TX_TIME_DAYS'].min())
        wk_abs  = day_min // 7 + 1
        wk_rel  = int(wk_abs - base_week + 1)
        marks.append((wk_rel, f"{y}-{m:02d}"))
    return marks


In [15]:
def evaluate_stream(df, features, scaler, clf, thr, granularity='week', base_week=None):
    import time, numpy as np, pandas as pd
    rows = []

    # Usa best_iter sólo si existe y es >= 0; si no, usa el modelo tal cual (todas las trees entrenadas)
    best_iter = getattr(clf, "_best_iter_internal", None)
    use_best = isinstance(best_iter, (int, np.integer)) and best_iter >= 0

    for key, chunk, label in group_chunks(df, granularity=granularity, start_date_str=START_DATE):
        
        if isinstance(key, int) and base_week is not None:
            pos = int(key) - int(base_week) + 1
        else:
            pos = int(key) if isinstance(key, int) else None

        X = scaler.transform(chunk[features].astype('float32').values)
        y = chunk[TARGET].astype('int32').values

        t0 = time.perf_counter()
        if use_best:
            try:
                p = clf.predict_proba(X, iteration_range=(0, best_iter + 1))[:, 1]
            except TypeError:
                p = clf.predict_proba(X, ntree_limit=best_iter + 1)[:, 1]
        else:
            p = clf.predict_proba(X)[:, 1]
        dt = time.perf_counter() - t0

        # métricas
        try:
            ap = float(average_precision_score(y, p)) if y.sum() > 0 else float('nan')
        except Exception:
            ap = float('nan')

        f1 = float(f1_score(y, (p >= thr).astype(int), zero_division=0))
        ms_per_tx = (dt / max(1, len(y))) * 1e3

        rows.append({"pos": pos, "label": label, "n": int(len(y)),
                     "AUPRC": ap, "F1": f1, "infer_ms_per_tx": ms_per_tx})

    metrics = pd.DataFrame(rows).sort_values('pos').reset_index(drop=True)
    return metrics


In [16]:
def build_adaptation_series(metrics_df, segments, window=8, start_weeks=2, mode="IAN"):
    """
    Devuelve una serie semanal 'AdaptIdx' en [0..1+] (capado) que se resetea por segmento.
    mode="IAN"  -> AUPRC / plateau_segment
    mode="PtP"  -> (AUPRC - start_level) / (plateau - start_level)
    """
    m = metrics_df.set_index('pos').sort_index()
    rows = []
    for seg in segments:
        s, e, lab = int(seg['start_pos']), int(seg['end_pos']), seg['label']
        sub = m.loc[m.index.intersection(range(s, e+1))].copy()
        if sub.empty:
            continue
        ap = sub['AUPRC'].values
        # Plateau: media de las últimas 'window' semanas del segmento
        w = max(1, min(window, len(ap)))
        plateau = float(np.nanmean(ap[-w:]))
        # Nivel de arranque: media de primeras 'start_weeks' semanas
        sw = max(1, min(start_weeks, len(ap)))
        start_level = float(np.nanmean(ap[:sw]))

        if mode.upper() == "IAN":
            denom = plateau if plateau != 0 else np.nan
            idx = (ap / denom) if denom and not np.isnan(denom) else np.full_like(ap, np.nan, dtype=float)
        else:  # PtP
            denom = (plateau - start_level)
            idx = ((ap - start_level) / denom) if (denom != 0 and not np.isnan(denom)) else np.full_like(ap, np.nan)

        # capamos para que sea legible (permitimos leve overshoot)
        idx = np.clip(idx, 0.0, 1.2)
        sub = sub.reset_index()
        sub['AdaptIdx'] = idx
        sub['segment']  = lab
        rows.append(sub[['pos','AdaptIdx','segment']])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=['pos','AdaptIdx','segment'])


def plot_adaptation_line(adapt_df, out_dir, month_markers=None, bimonth_markers=None, title="Índice de adaptación (semanal)"):
    if adapt_df.empty:
        return
    dfp = adapt_df.sort_values('pos')
    plt.figure()
    plt.plot(dfp['pos'].values, dfp['AdaptIdx'].values)
    # marcas de mes (grises) y bimestres (punteadas)
    if month_markers:
        for wk, _ in month_markers: plt.axvline(wk, color='gray', alpha=0.15)
        xs, labs = zip(*month_markers); plt.xticks(xs, labs, rotation=45, ha='right')
    if bimonth_markers:
        for wk, _ in bimonth_markers: plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)
    plt.axhline(1.0, color='gray', linestyle='--', alpha=0.5)  # línea del plateau
    plt.ylim(0, 1.2)
    plt.xlabel('Semanas'); plt.ylabel('Índice (0..1)')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'adaptacion_linea.png')); plt.close()


In [17]:
def schedule_to_week_segments(schedule_ym, eval_df: pd.DataFrame, base_week: int):
    """
    Convierte tu schedule por año/mes en segmentos en eje semanal re-baselineado.
    Devuelve lista de dicts: {'start_pos','end_pos','label'}
    - 'label' se arma como 'YYYY-MM..YYYY-MM (Sx)'
    - Si hay solapes en meses, los segmentos saldrán solapados; en la práctica tu eval no solapa.
    """
    segs = []
    for step in schedule_ym:
        y1, m1 = int(step['start']['year']), int(step['start']['month'])
        y2, m2 = int(step['end']['year']),   int(step['end']['month'])
        scen = int(step['scenario'])

        # filas de esos meses dentro del eval_df
        in_range = eval_df[
            ((eval_df['TX_YEAR'] >  y1) | ((eval_df['TX_YEAR']==y1) & (eval_df['TX_MONTH']>=m1))) &
            ((eval_df['TX_YEAR'] <  y2) | ((eval_df['TX_YEAR']==y2) & (eval_df['TX_MONTH']<=m2)))
        ]
        if in_range.empty: 
            continue

        day_min = int(in_range['TX_TIME_DAYS'].min())
        day_max = int(in_range['TX_TIME_DAYS'].max())
        wk1_abs = day_min // 7 + 1
        wk2_abs = day_max // 7 + 1
        wk1 = int(wk1_abs - base_week + 1)
        wk2 = int(wk2_abs - base_week + 1)

        lab = f"{y1}-{m1:02d}..{y2}-{m2:02d} (S{scen})"
        segs.append({'start_pos': wk1, 'end_pos': wk2, 'label': lab, 'scenario': scen})

    # Ordena y compacta si hay huecos/solapes leves
    segs = sorted(segs, key=lambda s: (s['start_pos'], s['end_pos']))
    return segs



In [18]:
schedule_ym = [
  # ======================
  # PRETRAIN (Ene–Abr 2025): S1 + S2 simultáneos
  # ======================
  {"scenario": 1, "start": {"year": 2025, "month": 1}, "end": {"year": 2025, "month": 4},
   "params": {"amount_threshold": 140}},
  {"scenario": 2, "start": {"year": 2025, "month": 1}, "end": {"year": 2025, "month": 4},
   "params": {"n_per_day": 3, "window_days": 21}},

  # ======================
  # Bimestres (S1 → S2 → S3) hasta 24 meses
  # ======================

  # Bloque 1 (May–Jun 2025): S1
  {"scenario": 1, "start": {"year": 2025, "month": 5}, "end": {"year": 2025, "month": 5},
   "params": {"amount_threshold": 130}},
   {"scenario": 1, "start": {"year": 2025, "month": 6}, "end": {"year": 2025, "month": 6},
   "params": {"amount_threshold": 130}},

  # Bloque 2 (Jul–Ago 2025): S2
  {"scenario": 2, "start": {"year": 2025, "month": 7}, "end": {"year": 2025, "month": 7},
   "params": {"n_per_day": 4, "window_days": 21}},

  # Bloque 2 (Jul–Ago 2025): S2
  {"scenario": 2, "start": {"year": 2025, "month": 8}, "end": {"year": 2025, "month": 8},
   "params": {"n_per_day": 4, "window_days": 21}},

  # Bloque 3 (Sep 2025): S3 (más suave)
  {"scenario": 3, "start": {"year": 2025, "month": 9}, "end": {"year": 2025, "month": 9},
   "params": {"n_customers_per_day": 15, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/3}},
  # Bloque 3 (Oct 2025): S3 (más intenso)
  {"scenario": 3, "start": {"year": 2025, "month": 10}, "end": {"year": 2025, "month": 10},
   "params": {"n_customers_per_day": 15, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/3}},

  # Bloque 4 (Nov–Dic 2025, NAVIDAD): S1 “suavizado”
  {"scenario": 1, "start": {"year": 2025, "month": 11}, "end": {"year": 2025, "month": 11},
   "params": {"amount_threshold": 150}},
  {"scenario": 1, "start": {"year": 2025, "month": 12}, "end": {"year": 2025, "month": 12},
   "params": {"amount_threshold": 180}},

  # Bloque 5 (Ene–Feb 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 1}, "end": {"year": 2026, "month": 2},
   "params": {"n_per_day": 4, "window_days": 21}},

  # Bloque 6 (Mar–Abr 2026): S3
  {"scenario": 3, "start": {"year": 2026, "month": 3}, "end": {"year": 2026, "month": 4},
   "params": {"n_customers_per_day": 4, "window_days": 14, "amp_factor": 5, "frac_to_flip": 1/3}},

  # Bloque 7 (May–Jun 2026): S1
  {"scenario": 1, "start": {"year": 2026, "month": 5}, "end": {"year": 2026, "month": 6},
   "params": {"amount_threshold": 130}},

  # Bloque 8 (Jul–Ago 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 7}, "end": {"year": 2026, "month": 7},
   "params": {"n_per_day": 4, "window_days": 28}},
   # Bloque 8 (Jul–Ago 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 8}, "end": {"year": 2026, "month": 8},
   "params": {"n_per_day": 4, "window_days": 28}},

  # Bloque 9 (Sep–Oct 2026): S3
  {"scenario": 3, "start": {"year": 2026, "month": 9}, "end": {"year": 2026, "month": 9},
   "params": {"n_customers_per_day": 15, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/3}},
  # Bloque 9 (Oct 2026): S3 (más intenso)
  {"scenario": 3, "start": {"year": 2026, "month": 10}, "end": {"year": 2026, "month": 10},
   "params": {"n_customers_per_day": 15, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/3}},

  # Bloque 10 (Nov–Dic 2026, NAVIDAD): S1 “suavizado”
  {"scenario": 1, "start": {"year": 2026, "month": 11}, "end": {"year": 2026, "month": 11},
   "params": {"amount_threshold": 150}},
  {"scenario": 1, "start": {"year": 2026, "month": 12}, "end": {"year": 2026, "month": 12},
   "params": {"amount_threshold": 180}},   # puedes subir a 180 en dic si quieres aún más “suavizado”
]


In [19]:
# =========================
# 1) Segmentos con scenario
# =========================
def schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week):
    segs = []
    for step in schedule_ym:
        y1,m1 = int(step['start']['year']), int(step['start']['month'])
        y2,m2 = int(step['end']['year']),   int(step['end']['month'])
        scen  = int(step['scenario'])

        in_range = eval_df[
            ((eval_df['TX_YEAR'] >  y1) | ((eval_df['TX_YEAR']==y1) & (eval_df['TX_MONTH']>=m1))) &
            ((eval_df['TX_YEAR'] <  y2) | ((eval_df['TX_YEAR']==y2) & (eval_df['TX_MONTH']<=m2)))
        ]
        if in_range.empty:
            continue

        dmin, dmax = int(in_range['TX_TIME_DAYS'].min()), int(in_range['TX_TIME_DAYS'].max())
        wk1_abs, wk2_abs = dmin//7 + 1, dmax//7 + 1
        wk1 = int(wk1_abs - base_week + 1)
        wk2 = int(wk2_abs - base_week + 1)
        lab = f"{y1}-{m1:02d}..{y2}-{m2:02d} (S{scen})"
        segs.append({'start_pos': wk1, 'end_pos': wk2, 'label': lab, 'scenario': scen})

    return sorted(segs, key=lambda s: (s['start_pos'], s['end_pos']))


# ============================================
# 2) Plateau local (cola del segmento, robusto)
# ============================================
def _calc_plateau_from_tail(ap, window='auto', method='percentile', q=0.80):
    """
    Plateau = estadístico de cola de la serie AUPRC del segmento.
    window: entero o 'auto' (~30% del largo, min 3)
    method: 'mean' | 'median' | 'percentile'
    q: percentil (si method='percentile'), p.ej. 0.80
    """
    import numpy as np
    L = len(ap)
    W = max(3, int(round(0.3*L))) if (isinstance(window, str) and window.lower()=='auto') else int(window)
    W = max(1, min(W, L))
    tail = np.asarray(ap[-W:], dtype=float)

    if method == 'median':
        plateau = float(np.nanmedian(tail))
    elif method == 'percentile':
        plateau = float(np.nanpercentile(tail, q*100.0))
    else:
        plateau = float(np.nanmean(tail))
    return plateau, W


def compute_segment_plateaux(metrics_df, segments, window='auto', method='percentile', q=0.80):
    """
    Devuelve DF: [start_pos, end_pos, label, scenario, len_weeks, plateau_local]
    """
    import pandas as pd, numpy as np
    m = metrics_df.set_index('pos').sort_index()
    rows = []
    for seg in segments:
        s, e = int(seg['start_pos']), int(seg['end_pos'])
        sub = m.loc[m.index.intersection(range(s, e+1))]
        if sub.empty:
            rows.append({**seg, 'len_weeks': 0, 'plateau_local': np.nan})
            continue
        ap = sub['AUPRC'].values
        plateau, _ = _calc_plateau_from_tail(ap, window=window, method=method, q=q)
        rows.append({**seg, 'len_weeks': int(len(ap)), 'plateau_local': float(plateau)})
    import pandas as pd
    return pd.DataFrame(rows).sort_values('start_pos').reset_index(drop=True)


# =============================================================
# 3) IAN combinado (memoria → actual) con rampa y actualización
# =============================================================
def build_adaptation_series_with_blend(
    metrics_df, seg_df,
    ramp_weeks=4, cap=(0.0, 1.2),
    update_strategy='ema', alpha=0.3, rolling_k=3
):
    """
    Genera serie semanal con:
      - AdaptIdx_local = AUPRC / plateau_local (IAN clásico por bloque)
      - AdaptIdx_blend = AUPRC / P_t, donde P_t hace rampa del plateau_ref_prev -> plateau_local
    Memoria por escenario:
      - 'ema'      : ref_post = (1-α)*ref_prev + α*plateau_local
      - 'replace'  : ref_post = plateau_local
      - 'rolling_k': ref_post = media de últimos K plateau_local de ese escenario
    """
    import numpy as np, pandas as pd, collections

    m = metrics_df.set_index('pos').sort_index()
    rows = []
    ref_plateau = {}  # memoria por scenario
    hist_plateaux = collections.defaultdict(list)

    for _, r in seg_df.sort_values('start_pos').iterrows():
        s, e   = int(r['start_pos']), int(r['end_pos'])
        scen   = int(r['scenario'])
        label  = r['label']
        P_loc  = float(r['plateau_local'])

        sub = m.loc[m.index.intersection(range(s, e+1))].copy()
        if sub.empty or not np.isfinite(P_loc) or P_loc <= 0:
            continue

        ap = sub['AUPRC'].values.astype(float)
        L  = len(ap)

        # IAN clásico (local)
        idx_local = ap / P_loc

        # Estándar dinámico P_t: memoria->actual con rampa
        P_prev = ref_plateau.get(scen, np.nan)
        if np.isfinite(P_prev) and P_prev > 0:
            t = np.arange(L, dtype=float)
            beta_t = np.clip(t / max(1, ramp_weeks), 0.0, 1.0)  # 0..1
            P_t = (1.0 - beta_t) * P_prev + beta_t * P_loc
        else:
            P_t = np.full(L, P_loc, dtype=float)  # primera aparición del escenario

        idx_blend = ap / P_t

        # Cap visual
        if cap is not None:
            lo, hi = cap
            if hi is None:
                idx_local = np.maximum(idx_local, lo)
                idx_blend = np.maximum(idx_blend, lo)
            else:
                idx_local = np.clip(idx_local, lo, hi)
                idx_blend = np.clip(idx_blend, lo, hi)

        sub = sub.reset_index()
        sub['segment']         = label
        sub['scenario']        = scen
        sub['AdaptIdx_local']  = idx_local
        sub['AdaptIdx_blend']  = idx_blend
        rows.append(sub[['pos','segment','scenario','AdaptIdx_local','AdaptIdx_blend']])

        # Actualizar memoria del escenario
        if update_strategy == 'ema':
            if np.isfinite(P_prev) and P_prev > 0:
                P_post = (1.0 - alpha) * P_prev + alpha * P_loc
            else:
                P_post = P_loc
            ref_plateau[scen] = float(P_post)

        elif update_strategy == 'replace':
            ref_plateau[scen] = P_loc

        elif update_strategy == 'rolling_k':
            hist_plateaux[scen].append(P_loc)
            last_k = hist_plateaux[scen][-int(rolling_k):]
            ref_plateau[scen] = float(np.mean(last_k))

        else:
            ref_plateau[scen] = P_loc  # fallback

    import pandas as pd
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=['pos','segment','scenario','AdaptIdx_local','AdaptIdx_blend']
    )


# ============================================
# 4) Plot de la curva combinada (opcional)
# ============================================
def plot_adapt_blend_line(adapt_df, out_dir, month_markers=None, bimonth_markers=None,
                          title="Índice de adaptación combinado (memoria → actual)"):
    import matplotlib.pyplot as plt, os
    if adapt_df.empty: return
    dfp = adapt_df.sort_values('pos')
    plt.figure()
    plt.plot(dfp['pos'].values, dfp['AdaptIdx_blend'].values)
    if month_markers:
        for wk, _ in month_markers: plt.axvline(wk, color='gray', alpha=0.15)
        xs, labs = zip(*month_markers); plt.xticks(xs, labs, rotation=45, ha='right')
    if bimonth_markers:
        for wk, _ in bimonth_markers: plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)
    plt.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    plt.ylim(0, 1.2); plt.xlabel('Semanas'); plt.ylabel('Índice (0..1)')
    plt.title(title); plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, 'adaptacion_blend_linea.png')); plt.close()


In [22]:
def add_terminal_rolling_features(df):
    # df debe estar ordenado por TX_DATETIME
    g = df.sort_values('TX_DATETIME').groupby('TERMINAL_ID', sort=False)

    # conteos acumulados
    df['term_tx_cum']   = g.cumcount() + 1
    df['term_frd_cum']  = g['TX_FRAUD'].cumsum().shift(1).fillna(0)  # hasta t-1
    df['term_tx_cum_m1']= df['term_tx_cum'] - 1

    # tasa acumulada suavizada (prior= prevalencia global aprox.)
    prior = df['TX_FRAUD'].mean()
    k     = 20  # fuerza de suavizado
    df['term_fraud_rate_cum'] = (df['term_frd_cum'] + k*prior) / (df['term_tx_cum_m1'] + k)

    # rolling por ventana (requiere índice por terminal con frecuencia; si no, usa EWM “por llegada”)
    df['term_tx_ewm']   = g['TX_FRAUD'].apply(lambda s: s.shift(1).ewm(alpha=0.2, adjust=False).mean()).reset_index(level=0, drop=True)
    # cantidad desde la última semana (aprox. 7d) — si no tienes gaps regulares, usa diferencia de fechas
    df['term_time_since_prev'] = g['TX_DATETIME'].diff().dt.total_seconds().fillna(1e9)

    # limpia temporales
    df = df.drop(columns=['term_tx_cum_m1'])
    return df


In [ ]:
print("Cargando datos...")
df = load_all_parquet_by_year(BASE_PATH)
df = add_time_indexes(df)


# Split pretraining (meses 1..4)
train_mask, val_mask = split_pretrain_val(df, pre_months=PRE_MONTHS, val_days_last_month=VAL_DAYS_LAST_MONTH)
train_df = df[train_mask]
val_df   = df[val_mask]



# Entrenar baseline
print("Entrenando baseline (LR) con meses 1..4...")
features = [c for c in FEATURES if c in df.columns]
scaler, clf, thr, ap_ref, f1_ref, train_time_s, inf_time_per_tx_val = fit_baseline(train_df, val_df, features)

# Serie mensual desde el mes 5 (o desde donde quieras evaluar)
print("Evaluando serie temporal...")
eval_df = df[df['_month_idx'] >= (PRE_MONTHS + 1)]
#print(eval_df['TX_MONTH'].value_counts().sort_index())
#print(eval_df['TX_YEAR'].value_counts().sort_index())
base_week = compute_base_week(eval_df)   
metrics = evaluate_stream(eval_df, features, scaler, clf, thr, granularity='week',base_week=base_week)

    # Guardar métricas crudas
metrics.to_csv(os.path.join(OUT_DIR, 'monthly_metrics.csv'), index=False)

month_marks = compute_month_markers(eval_df,base_week)
bimonth_marks  = compute_bimonth_markers(eval_df, pre_months=PRE_MONTHS, base_week=base_week)

print(month_marks)
print(bimonth_marks)
# Olvido (MaxDrawdown%)
mdd = compute_max_drawdown_percent(metrics['AUPRC'].values)

#
dd_df = drawdown_series(metrics)
dd_df.to_csv(os.path.join(OUT_DIR, 'drawdown_semana.csv'), index=False)
plot_drawdown_weekly(dd_df, OUT_DIR, month_markers=month_marks, bimonth_markers=bimonth_marks)
#max_dd_pct = compute_max_drawdown_percent_from_series(dd_df)


# # CALCULO ÍNDICE DE ADAPTACIÓN
segments = schedule_to_week_segments(schedule_ym, eval_df, base_week)

# Cada segmento es un patrón de 2 meses (aprox. 8–9 semanas
adapt_ian = build_adaptation_series(metrics, segments, window=4, start_weeks=2, mode="IAN")
plot_adaptation_line(adapt_ian, OUT_DIR, month_markers=month_marks, bimonth_markers=bimonth_marks,
                     title="Índice de adaptación normalizado (AUPRC/plateau)")



#CALCULO DEL INDICE DE ADAPTACIÓN COMBINADO (MEMORIA → ACTUAL)
# 1) Segmentar usando scenario
segments1 = schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week)

# 2) Plateau local por bloque (bimestral: window=8 suele ir bien; o 'auto')
seg_plateaux = compute_segment_plateaux(
    metrics, segments1, window=8, method='percentile', q=0.80
)

# 3) Serie IAN combinada (memoria → actual) con rampa de 4 semanas y EMA(α=0.3)
adapt_blend = build_adaptation_series_with_blend(
    metrics, seg_plateaux,
    ramp_weeks=4, cap=(0.0, 1.2),
    update_strategy='ema', alpha=0.3, rolling_k=3
)

# 4) Guardar y graficar
adapt_blend.to_csv(os.path.join(OUT_DIR, 'adaptacion_blend_semana.csv'), index=False)
plot_adapt_blend_line(adapt_blend, OUT_DIR, month_markers=month_marks, bimonth_markers=bimonth_marks)



    # Gráficas
plot_series(metrics, OUT_DIR, month_marks,bimonth_marks)

    # Resumen
print("\n=== RESUMEN BASELINE #1 ===")
print(f"AUPRC de referencia (val mes 4): {ap_ref:.4f}")
print(f"F1 de referencia (val mes 4)   : {f1_ref:.4f} @thr={thr:.3f}")
print(f"Tiempo de entrenamiento (s)     : {train_time_s:.2f}")
print(f"Inferencia val (ms/tx aprox)    : {inf_time_per_tx_val*1e3:.3f}")
print(f"MaxDrawdown% (olvido)           : {mdd:.2f}%")


print(f"\nArchivos guardados en: {OUT_DIR}")
print(" - monthly_metrics.csv")
print(" - tta90_por_regimen.csv")
print(" - auprc_tiempo.png, f1_tiempo.png, auprc_drawdown.png, latencia_tiempo.png, tta90_regimenes.png")

Cargando datos...


KeyError: 'TX_DATETIME'